In [6]:
%pwd

'c:\\Users\\Asus\\Machine_learning\\LLM\\Language_Model\\GPT_from_scratch\\notebook'

In [7]:
import os 

In [8]:
os.chdir("..\.")

In [9]:
%pwd

'c:\\Users\\Asus\\Machine_learning\\LLM\\Language_Model\\GPT_from_scratch'

In [10]:
import torch
from torch import nn
import torch.nn.functional as F

## Version 4: Setf Attention 

In [11]:
torch.manual_seed(1337) 
B,T,C   = 4,8,32                        # batch size, block size, embedding dimension
x       = torch.randn(B,T,C) 

tril    = torch.tril(torch.ones(T,T))   # (T,T) lower triangular matrix
wei     = torch.zeros (T,T)             # (T,T) weight matrix initialized to zeros
wei     = wei.masked_fill(tril==0, float('-inf')) # fill upper 
wei     = F.softmax(wei,dim=-1)         # apply softmax to get attention weights
out     = wei @ x                       # (T,T) @ (B,T,C) ==> (B,T,C)

- wei = torch.zeros(T,T) ==> initialize affinities between all the different tokens or nodes to be zeros. 
- But, we dont want **wei =torch.zeros(T,T)** to be all uniform. 
    - Becasue different tokens will find  other tokens more or less intresting other tokens. **We want that to be data dependent.** 
        - Example:  Vowel maybe looking for consonates from the past.So i want to flow that information to cuurrent token. But want to be data dependent way. 

## This is the problem Self Attention  Solves**

- The way Self attention solves is follows.
- The every single token at each position will emit 2 vectors, **1. Query and 2. Key** 
    - 1. **Query    : what im looking For**
    - 2. **Key      : What Im offering or what do i contain**
    - 3. **Value    : What i communicate to You.**
- By doing dot product of **Query & Key** we get the affinities between tokens in a sequence. 
- The dot product between Query and key becomes the weight (wei).  If key and query is producing high value, then it's telling that we need to learn more about that specific token as supposed to any other token in the sequence. 

In [12]:
## Implimentation of single head of Self Attention
torch.manual_seed(1337)
B,T,C   = 4,8,32                        # batch size, block size, embedding dimension
x       = torch.randn(B,T,C)    

# single head of self attention
head_size   = 16 
key         = nn.Linear(C,head_size,bias=False) # what each token contains 
query       = nn.Linear(C,head_size,bias=False) # what each token wants to pay attention to     
value       = nn.Linear(C,head_size,bias=False) # what information each token offer to share 
# key & query are parallel linear layers that transform the input x into key and query vectors. (No communication between k and q)
k           = key(x)                    # ==> (B,T,head_size) = # (4,8,16) 
q           = query(x)                  # ==> (B,T,head_size) = # (4,8,16)
v           = value(x)                  # ==> (B,T,head_size) = # (4,8,16)
# next step is to make the k & q vectors interact with each other to compute the attention weights.
wei     = q @ k.transpose(-2,-1)      # (B,T,16) @ (B,16,T) ==> (B,T,T)   
wei     = wei * head_size ** - 0.5
# The k and q vectors are used to compute the attention weights.

tril    = torch.tril(torch.ones(T,T))   # (T,T) lower triangular matrix

wei     = wei.masked_fill(tril==0, float('-inf')) # fill upper  
wei     = F.softmax(wei,dim=-1)         # apply softmax to get attention weights
v       = value(x)
out     = wei @ v                       # (B,T,T) @ (B,T,head_size) ==> (B,T,head_size) 
#out     = wei @ x                       # (T,T) @ (B,T,C) ==> (B,T,C)   

In [13]:
out.shape,wei.shape

(torch.Size([4, 8, 16]), torch.Size([4, 8, 8]))



Notes:

- Attention is a communication mechanism. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with tril, allowing all tokens to communicate. 
    - ```
            tril    = torch.tril(torch.ones(T,T))   # (T,T) lower triangular matrix
            
            wei     = F.softmax(wei,dim=-1)         # apply softmax to get attention weights

- This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
    - ```
        tril    = torch.tril(torch.ones(T,T))   # (T,T) lower triangular matrix
        wei     = wei.masked_fill(tril==0, float('-inf')) # fill upper  
        wei     = F.softmax(wei,dim=-1)         # apply softmax to get attention weights

## Self-attention
Just means that the keys and values are produced from the same source as queries. 
- The reason why its called **Self Attention** 
    - The key and query is produce from common source, that is **x**. So tokens are self attending


## Cross-attention
The queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- For this case, the reason behind name **Cross Attention** is the key, query and value is from different source. 
- For example: From Encoder decoder transformers, The query are produced from **x** but the key and value are produced from seperated external source, ie sometimes from encoder block.  

## Scaled
-  attention additional divides wei by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

- value(x) creates the data that will actually be passed between tokens, weighted by how much attention each token pays to others (from q @ k^T). It’s essential to separate the concern of “how much to attend” (q, k) from “what to gather” (v).

In [14]:
k = torch.randn(B,T,head_size)  # (B,T,head_size)
q = torch.randn(B,T,head_size)  # (B,T,head_size)
wei = q @ k.transpose(-2,-1)  # (B,T,head_size) @ (B,head_size,T) ==> (B,T,T)

In [15]:
k.var(),q.var(),wei.var() 

(tensor(1.0449), tensor(1.0700), tensor(17.4690))

In [16]:
(wei * head_size**-0.5).var() 

tensor(1.0918)

In [17]:
wei = q @ k.transpose(-2,-1) * head_size**-0.5 # Normalize the attention weights

# normalization just used to control the varience of wei at initialization

In [18]:
## Sample 

torch.softmax(torch.tensor([0.1,-0.2,0.3,-0.2,0.5]),dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [19]:
torch.softmax(torch.tensor([0.1,-0.2,0.3,-0.2,0.5])*8,dim=-1)
# by multiplying with a constant, for this case mutiply with 8, it sharpen the distribution
# its sharpenn towords the max value, and make the other values smaller

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

### Sentimental Analysis 

- In sentimental Analysis the all token should communicate to each other (previous and future tokens). 
- In this case we use **Encoder block** of self attention. 

## self.register_buffer()

In [20]:
import torch
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        # Register a buffer named 'my_constant_tensor'
        # This tensor will be saved with the model but won't be trained.
        self.register_buffer('my_constant_tensor', torch.tensor([1.0, 2.0, 3.0]))

        # A regular trainable parameter
        self.linear = nn.Linear(3, 1)

    def forward(self, x):
        # You can use the buffer in your forward pass
        x = x * self.my_constant_tensor
        return self.linear(x)

In [21]:
model = MyModel()
print("Model parameters:", list(model.parameters()))
print("Model buffers:", list(model.buffers()))

Model parameters: [Parameter containing:
tensor([[ 0.4163,  0.2083, -0.5005]], requires_grad=True), Parameter containing:
tensor([0.2631], requires_grad=True)]
Model buffers: [tensor([1., 2., 3.])]


**register_buffer(name, tensor)* is used to:

- Add a non-trainable tensor to your model that is still part of the model’s state (saved/loaded, moved to device).

- It’s perfect for masks, running stats, or fixed embeddings.